This notebook produces every number and figure in the post
[Learning Rate, Batch Size, Dropout](lr-batch-size-dropout.md).

**Starting point.** The framework is adapted from the course notebook
*Hyperparameters in Deep Learning — a hands-on tour with PyTorch Lightning*
(`hyperparameters_lightning.ipynb`): its `HParams` config, `FashionMNISTDataModule`, `SmallCNN`
and `LitClassifier` are reused. Every change is marked in the code with `# changed:` or
`# added:`. The starter's quick one-seed sweeps are replaced by a controlled experiment:

- one hyperparameter changed at a time around a fixed baseline;
- every configuration run with **three seeds**, all on the **same validation split**;
- a batch-size sweep repeated with the learning rate scaled in proportion (the starter's exercise 3);
- a training subset re-scored **in eval mode** every epoch, so dropout cannot hide the generalisation gap.

**Running it (Google Colab).**

1. *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. The first cell installs
   Lightning, asks once for Google Drive access, and stops with a clear message if no GPU is
   attached.
2. The sweep is **resumable**. Every finished run saves its weights and learning curve to
   `MyDrive/hyperparameters_lightning/results/`. If the session disconnects or resets, run all
   cells again and it continues where it stopped. Every run is seeded, so the result is the
   same as an uninterrupted run.
3. The last cell zips the result tables and figures and downloads them. Also save the notebook
   with its outputs (*File → Download → Download .ipynb*).

GPU results are deterministic on the same hardware. On a different GPU or on CPU, the numbers
differ slightly.

## 1. Setup

In [ ]:
import os
import sys
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # added: deterministic cuBLAS

IN_COLAB = "google.colab" in sys.modules
SAVE_TO_DRIVE = IN_COLAB   # keep results in Google Drive, so a Colab reset cannot lose finished runs
if IN_COLAB:
    %pip install -q lightning      # torch, torchvision, pandas, matplotlib are preinstalled on Colab

import logging
import platform
import time
import warnings
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms

import lightning as L

for name in ("lightning.pytorch", "lightning.fabric"):
    logging.getLogger(name).setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*does not have many workers.*")
warnings.filterwarnings("ignore", message=".*LeafSpec.*")

OUT = Path(".")
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")                   # asks once for permission
    OUT = Path("/content/drive/MyDrive/hyperparameters_lightning")
RESULTS = OUT / "results"; RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES = OUT / "figures"; FIGURES.mkdir(parents=True, exist_ok=True)
print("results and figures are saved under", OUT.resolve())

assert torch.cuda.is_available(), (
    "No GPU found. In Colab: Runtime > Change runtime type > T4 GPU, then Runtime > Run all.")
print(f"python {platform.python_version()} | torch {torch.__version__} | "
      f"torchvision {torchvision.__version__} | lightning {L.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)} | CUDA {torch.version.cuda} | Colab: {IN_COLAB}")

## 2. The starter framework

### `HParams`: every hyperparameter in one place

Unchanged from the starter except for two added fields. `split_seed` fixes the train/validation
split independently of the training seed, so every run is validated on the same images.
`probe_size` sets how many training images are re-scored each epoch.

In [ ]:
@dataclass
class HParams:
    # ---- Data -------------------------------------------------------------
    data_dir: str = "./data"
    batch_size: int = 64          # samples per gradient step
    val_fraction: float = 0.1     # share of the training set held out for validation
    train_subset: Optional[int] = None  # use only N training images (None = all)
    augment: bool = False         # random crop + horizontal flip on training images
    num_workers: int = 0          # data-loading processes (0 is safest in notebooks)
    split_seed: int = 42          # added: fixes the train/val split for every run
    probe_size: int = 10_000      # added: training images re-scored in eval mode each epoch

    # ---- Model ------------------------------------------------------------
    conv1_channels: int = 32
    conv2_channels: int = 64
    hidden_dim: int = 128
    activation: str = "relu"      # relu | leaky_relu | gelu | tanh
    use_batchnorm: bool = True    # batch normalisation after each conv
    dropout: float = 0.25         # dropout probability in the classifier head

    # ---- Optimisation -----------------------------------------------------
    optimizer: str = "adam"       # sgd | adam | adamw
    lr: float = 1e-3
    momentum: float = 0.9         # used by SGD only
    weight_decay: float = 0.0
    scheduler: str = "none"       # none | step | cosine

    # ---- Training loop ----------------------------------------------------
    max_epochs: int = 5
    patience: int = 3             # (early stopping is not used in this experiment)
    seed: int = 42                # weight init, shuffling and dropout masks

### Data module

The starter's `FashionMNISTDataModule` with three changes:

1. The split uses `split_seed` instead of `seed`.
2. A second validation loader serves the probe: the first `probe_size` training images, with no augmentation.
3. With augmentation off, every epoch sees identical tensors. So the images are converted once,
   with the same `ToTensor` + `Normalize` arithmetic done in bulk, instead of image by image on
   every step. This is about 20× faster and gives the same numbers.

In [ ]:
CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
FMNIST_MEAN, FMNIST_STD = (0.2860,), (0.3530,)


def as_tensors(ds):
    # added: bulk version of ToTensor() + Normalize() for an (optionally subset) FashionMNIST
    base, idx = (ds.dataset, ds.indices) if isinstance(ds, Subset) else (ds, slice(None))
    x = base.data[idx].float().div(255).unsqueeze(1)
    return TensorDataset((x - FMNIST_MEAN[0]) / FMNIST_STD[0], base.targets[idx])


class FashionMNISTDataModule(L.LightningDataModule):
    def __init__(self, hp: HParams):
        super().__init__()
        self.hp = hp
        normalize = transforms.Normalize(FMNIST_MEAN, FMNIST_STD)
        self.eval_transform = transforms.Compose([transforms.ToTensor(), normalize])
        augmentations = []
        if hp.augment:
            augmentations = [transforms.RandomCrop(28, padding=2), transforms.RandomHorizontalFlip()]
        self.train_transform = transforms.Compose([*augmentations, transforms.ToTensor(), normalize])

    def prepare_data(self):
        datasets.FashionMNIST(self.hp.data_dir, train=True, download=True)
        datasets.FashionMNIST(self.hp.data_dir, train=False, download=True)

    def setup(self, stage=None):
        train_view = datasets.FashionMNIST(self.hp.data_dir, train=True, transform=self.train_transform)
        eval_view = datasets.FashionMNIST(self.hp.data_dir, train=True, transform=self.eval_transform)

        n_total = len(train_view)
        n_val = int(n_total * self.hp.val_fraction)
        generator = torch.Generator().manual_seed(self.hp.split_seed)   # changed: was hp.seed
        indices = torch.randperm(n_total, generator=generator).tolist()
        val_idx, train_idx = indices[:n_val], indices[n_val:]
        if self.hp.train_subset is not None:
            train_idx = train_idx[: self.hp.train_subset]

        self.train_ds = Subset(train_view, train_idx)
        self.val_ds = Subset(eval_view, val_idx)
        self.probe_ds = Subset(eval_view, train_idx[: self.hp.probe_size])          # added
        self.test_ds = datasets.FashionMNIST(self.hp.data_dir, train=False, transform=self.eval_transform)
        if not self.hp.augment:                                                      # added
            self.train_ds, self.val_ds, self.probe_ds, self.test_ds = map(
                as_tensors, (self.train_ds, self.val_ds, self.probe_ds, self.test_ds))

    def _loader(self, dataset, batch_size, shuffle):
        return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                          num_workers=self.hp.num_workers, persistent_workers=self.hp.num_workers > 0)

    def train_dataloader(self):
        return self._loader(self.train_ds, self.hp.batch_size, shuffle=True)

    def val_dataloader(self):
        # changed: [validation, probe] instead of validation only
        return [self._loader(self.val_ds, 512, shuffle=False), self._loader(self.probe_ds, 512, shuffle=False)]

    def test_dataloader(self):
        return self._loader(self.test_ds, 512, shuffle=False)

### Model and `LightningModule` (unchanged from the starter)

In [ ]:
ACTIVATIONS = {"relu": nn.ReLU, "leaky_relu": nn.LeakyReLU, "gelu": nn.GELU, "tanh": nn.Tanh}


class SmallCNN(nn.Module):
    def __init__(self, hp: HParams, num_classes: int = 10):
        super().__init__()
        act = ACTIVATIONS[hp.activation]
        c1, c2 = hp.conv1_channels, hp.conv2_channels

        def conv_block(in_ch, out_ch):
            layers = [nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)]
            if hp.use_batchnorm:
                layers.append(nn.BatchNorm2d(out_ch))
            layers += [act(), nn.MaxPool2d(2)]
            return nn.Sequential(*layers)

        self.features = nn.Sequential(conv_block(1, c1), conv_block(c1, c2))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(hp.dropout),
            nn.Linear(c2 * 7 * 7, hp.hidden_dim),
            act(),
            nn.Dropout(hp.dropout),
            nn.Linear(hp.hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


class LitClassifier(L.LightningModule):
    def __init__(self, hp: HParams):
        super().__init__()
        self.hp = hp
        self.save_hyperparameters(asdict(hp))
        self.model = SmallCNN(hp)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, x):
        return self.model(x)

    def _shared_step(self, batch, stage: str):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log(f"{stage}_acc", acc, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        self._shared_step(batch, "val")

    def test_step(self, batch, batch_idx):
        self._shared_step(batch, "test")

    def configure_optimizers(self):
        hp = self.hp
        params = self.parameters()
        if hp.optimizer == "sgd":
            optimizer = torch.optim.SGD(params, lr=hp.lr, momentum=hp.momentum, weight_decay=hp.weight_decay)
        elif hp.optimizer == "adam":
            optimizer = torch.optim.Adam(params, lr=hp.lr, weight_decay=hp.weight_decay)
        elif hp.optimizer == "adamw":
            optimizer = torch.optim.AdamW(params, lr=hp.lr, weight_decay=hp.weight_decay)
        else:
            raise ValueError(f"unknown optimizer: {hp.optimizer}")
        if hp.scheduler == "none":
            return optimizer
        if hp.scheduler == "step":
            scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(1, hp.max_epochs // 3), gamma=0.1)
        elif hp.scheduler == "cosine":
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=hp.max_epochs)
        else:
            raise ValueError(f"unknown scheduler: {hp.scheduler}")
        return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "interval": "epoch"}}

### Added: per-epoch bookkeeping

`TrackedClassifier` inherits the model, loss and optimizer from `LitClassifier` and only
changes how metrics are collected. Each epoch produces exactly one row:

- `train_*`: averaged over the epoch's mini-batches, **with dropout active**, while the weights change. This is the number a progress bar shows.
- `probe_*`: the fixed training subset, scored in eval mode (no dropout) after the epoch.
- `val_*`: the validation split, scored in eval mode after the epoch.
- `train_time`: wall-clock seconds for the training part of the epoch only.

Loss and correct-count sums stay on the GPU, and sample counts stay in Python, so no training
step forces a GPU→CPU synchronisation.

In [ ]:
class TrackedClassifier(LitClassifier):
    def __init__(self, hp: HParams):
        super().__init__(hp)
        self.history = []

    def _sync(self):
        if self.device.type == "cuda":
            torch.cuda.synchronize()

    def on_train_epoch_start(self):
        self._sync()
        self._t0 = time.perf_counter()
        self._train, self._n_train = torch.zeros(2, device=self.device), 0

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        self._train += torch.stack([loss.detach() * len(y), (logits.argmax(1) == y).sum()])
        self._n_train += len(y)
        return loss

    def on_validation_epoch_start(self):
        self._sync()
        self._train_time = time.perf_counter() - self._t0
        self._eval, self._n_eval = torch.zeros(2, 2, device=self.device), [0, 0]   # rows: val, probe

    def validation_step(self, batch, batch_idx, dataloader_idx=0):
        x, y = batch
        logits = self(x)
        self._eval[dataloader_idx] += torch.stack([
            nn.functional.cross_entropy(logits, y, reduction="sum"), (logits.argmax(1) == y).sum()])
        self._n_eval[dataloader_idx] += len(y)

    def on_train_epoch_end(self):               # runs after this epoch's validation pass
        (tl, tc), (vl, vc), (pl_, pc) = self._train.tolist(), *self._eval.tolist()
        tn, (vn, pn) = self._n_train, self._n_eval
        self.history.append(dict(
            epoch=self.current_epoch + 1,
            train_loss=tl / tn, train_acc=tc / tn,
            probe_loss=pl_ / pn, probe_acc=pc / pn,
            val_loss=vl / vn, val_acc=vc / vn,
            train_time=self._train_time,
        ))


def train_run(hp: HParams):
    # Adapted from the starter's fit(): no logger, no early stopping, deterministic kernels.
    L.seed_everything(hp.seed, workers=True, verbose=False)
    dm = FashionMNISTDataModule(hp)
    model = TrackedClassifier(hp)       # same seed -> same initial weights for every configuration
    trainer = L.Trainer(
        max_epochs=hp.max_epochs, accelerator="auto", devices=1, deterministic=True,
        logger=False, enable_checkpointing=False, enable_progress_bar=False,
        enable_model_summary=False, num_sanity_val_steps=0,
    )
    trainer.fit(model, datamodule=dm)
    return pd.DataFrame(model.history), model, dm

## 3. Experiment design

**Baseline.** The starter's `SmallCNN` with its default width (32, 64, 128) and default dropout
(0.25). Two deliberate changes from the starter's defaults:

- **SGD with momentum 0.9 instead of Adam.** The linear scaling rule tested here was derived for
  SGD. Adam rescales every step per parameter, so it would blur the effect being studied.
- **Batch normalisation off.** BatchNorm normalises with statistics of the current batch, so
  changing the batch size would also change the model's forward pass, not just the gradient
  noise. It also makes training much less sensitive to the learning rate, which would mute the
  learning-rate sweep.

Each sweep changes one field of `HParams`. The scaled batch-size sweep changes `batch_size` and
`lr` together on purpose, because their ratio is the variable being tested.

In [ ]:
SEEDS = [0, 1, 2]
BASE = replace(HParams(), optimizer="sgd", lr=0.01, momentum=0.9, use_batchnorm=False,
               dropout=0.25, batch_size=64, max_epochs=15)

SWEEPS = {
    "learning_rate":        [replace(BASE, lr=v) for v in [0.001, 0.003, 0.01, 0.03, 0.1, 0.3]],
    "batch_size":           [replace(BASE, batch_size=b) for b in [16, 64, 256, 1024]],
    "batch_size_scaled_lr": [replace(BASE, batch_size=b, lr=BASE.lr * b / 64) for b in [16, 64, 256, 1024]],
    "dropout":              [replace(BASE, dropout=p) for p in [0.0, 0.25, 0.5, 0.75]],
}

def cfg_id(hp):
    return f"lr={hp.lr:g}|bs={hp.batch_size}|p={hp.dropout:g}"

UNIQUE = {cfg_id(h): h for sweep in SWEEPS.values() for h in sweep}     # the baseline is shared
EPOCHS = BASE.max_epochs
print(f"{len(UNIQUE)} unique configurations x {len(SEEDS)} seeds = {len(UNIQUE) * len(SEEDS)} runs, "
      f"{EPOCHS} epochs each")
for name, sweep in SWEEPS.items():
    print(f"  {name:<22}", [cfg_id(h) for h in sweep])
print(f"\ntrainable parameters: {sum(p.numel() for p in SmallCNN(BASE).parameters()):,}")
pd.Series(asdict(BASE), name="baseline").to_frame()

## 4. Data check

Fashion-MNIST ships with `torchvision`: 60,000 training and 10,000 test images (28×28
grayscale, 10 classes). `val_fraction = 0.1` holds out **6,000 validation images** and leaves
**54,000 for training**. The 10,000 test images are not used until section 8.

In [ ]:
dm = FashionMNISTDataModule(BASE)
dm.prepare_data()
dm.setup()
print(f"train: {len(dm.train_ds):,}   val: {len(dm.val_ds):,}   probe: {len(dm.probe_ds):,}   test: {len(dm.test_ds):,}")

# The bulk conversion must match the starter's per-image transform exactly.
eval_view = datasets.FashionMNIST(BASE.data_dir, train=True, transform=dm.eval_transform)
val_idx = torch.randperm(60_000, generator=torch.Generator().manual_seed(BASE.split_seed))[:6_000]
x0, y0 = eval_view[int(val_idx[0])]
assert torch.allclose(dm.val_ds[0][0], x0, atol=1e-6) and dm.val_ds[0][1] == y0
print("bulk conversion matches ToTensor() + Normalize(): OK")

## 5. Run every configuration

The loop is **resumable**. After each run it saves the final weights to `results/weights/`
(needed only for the single test evaluation in section 8) and appends the run's learning curve
to `results/runs.csv`. Runs already present in both are skipped. To start from scratch, delete
the `results/` folder.

In [ ]:
RUNS_CSV = RESULTS / "runs.csv"
WEIGHTS = RESULTS / "weights"; WEIGHTS.mkdir(exist_ok=True)
PROGRESS = RESULTS / "progress.log"                 # one line per finished run


def weight_file(cid, seed):
    return WEIGHTS / f"{cid.replace('|', '_')}_seed{seed}.pt"


done = set()
if RUNS_CSV.exists():
    prev = pd.read_csv(RUNS_CSV)
    done = {(c, s) for c, s in zip(prev.cfg, prev.seed) if weight_file(c, s).exists()}
print(f"{len(done)} of {len(UNIQUE) * len(SEEDS)} runs already finished")

t_start = time.perf_counter()
for i, (cid, hp_c) in enumerate(UNIQUE.items(), 1):
    for seed in SEEDS:
        if (cid, seed) in done:
            continue
        t0 = time.perf_counter()
        hist, model, _ = train_run(replace(hp_c, seed=seed))
        torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, weight_file(cid, seed))
        hist = hist.assign(cfg=cid, seed=seed, lr=hp_c.lr, batch_size=hp_c.batch_size, dropout=hp_c.dropout)
        hist.to_csv(RUNS_CSV, mode="a", header=not RUNS_CSV.exists(), index=False)   # weights first, then CSV
        last = hist.iloc[-1]
        line = (f"[{i:2d}/{len(UNIQUE)}] {cid:<26} seed {seed}: val_acc {last.val_acc:.4f} "
                f"val_loss {last.val_loss:.4f} ({time.perf_counter() - t0:5.1f} s)")
        print(line)
        with PROGRESS.open("a") as f:
            f.write(line + "\n")
print(f"\ntraining time in this session: {(time.perf_counter() - t_start) / 60:.1f} min")

runs = pd.read_csv(RUNS_CSV).drop_duplicates(["cfg", "seed", "epoch"], keep="last")
n_done = runs[["cfg", "seed"]].drop_duplicates().shape[0]
assert n_done == len(UNIQUE) * len(SEEDS), f"only {n_done} runs finished; run this cell again"
print(f"{n_done} runs x {EPOCHS} epochs loaded from {RUNS_CSV}; "
      f"total training time {runs.train_time.sum() / 60:.1f} min")

## 6. Results table

One row per configuration. Every metric is the **mean ± standard deviation over the three
seeds**, taken at the final epoch. "Best-epoch" is the highest validation accuracy reached at any
epoch, averaged over seeds. The **gap** is probe accuracy minus validation accuracy, both
measured in eval mode. `s/epoch` is the median training time per epoch.

In [ ]:
final = runs[runs.epoch == EPOCHS]
agg = final.groupby("cfg").agg(
    lr=("lr", "first"), batch_size=("batch_size", "first"), dropout=("dropout", "first"),
    val_acc=("val_acc", "mean"), val_acc_sd=("val_acc", "std"),
    val_loss=("val_loss", "mean"), val_loss_sd=("val_loss", "std"),
    probe_acc=("probe_acc", "mean"), train_acc=("train_acc", "mean"),
)
agg["best_val_acc"] = runs.groupby(["cfg", "seed"]).val_acc.max().groupby("cfg").mean()
agg["gap"] = agg.probe_acc - agg.val_acc
agg["s_per_epoch"] = runs.groupby("cfg").train_time.median()


def sweep_table(name):
    rows = agg.loc[[cfg_id(h) for h in SWEEPS[name]]]
    return pd.DataFrame({
        "lr": rows.lr.map("{:g}".format), "batch": rows.batch_size, "dropout": rows.dropout,
        "val acc (%)": [f"{m*100:.2f} ± {s*100:.2f}" for m, s in zip(rows.val_acc, rows.val_acc_sd)],
        "val loss": [f"{m:.3f} ± {s:.3f}" for m, s in zip(rows.val_loss, rows.val_loss_sd)],
        "best-epoch val acc (%)": (rows.best_val_acc * 100).round(2),
        "train acc, dropout on (%)": (rows.train_acc * 100).round(2),
        "probe acc (%)": (rows.probe_acc * 100).round(2),
        "gap (pp)": (rows.gap * 100).round(2),
        "s/epoch": rows.s_per_epoch.round(2),
    }).reset_index(drop=True)


tables = {name: sweep_table(name) for name in SWEEPS}
for name, t in tables.items():
    print(f"\n=== {name} ===")
    print(t.to_string(index=False))
agg.to_csv(RESULTS / "summary.csv")

## 7. Figures

In [ ]:
from matplotlib.lines import Line2D

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 160, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
ep = np.arange(1, EPOCHS + 1)
CHANCE = np.log(10)     # cross-entropy of a uniform guess over 10 classes


def band(ax, cid, metric, color, label=None, ls="-"):
    # mean over seeds as a line, min-max over seeds as a shaded band
    g = runs[runs.cfg == cid].groupby("epoch")[metric]
    ax.plot(ep, g.mean(), color=color, ls=ls, lw=1.8, label=label)
    ax.fill_between(ep, g.min(), g.max(), color=color, alpha=0.18, lw=0)


def palette(n):
    return plt.cm.viridis(np.linspace(0.05, 0.85, n))


def loss_axes(ax, top=3.2):
    ax.set_yscale("log")
    ax.set_ylim(0.1, top)
    ticks = [t for t in [0.1, 0.2, 0.3, 0.5, 1, 2] if t < top]
    ax.set_yticks(ticks, labels=[f"{t:g}" for t in ticks])
    ax.minorticks_off()
    if top > CHANCE:
        ax.axhline(CHANCE, color="grey", ls=":", lw=1)
        ax.text(EPOCHS, CHANCE * 1.06, "chance (ln 10)", ha="right", va="bottom", fontsize=8, color="grey")
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))


def save(fig, name):
    fig.savefig(FIGURES / name, bbox_inches="tight", facecolor="white")
    plt.show()

### Figure 1: learning rate

Training loss (with dropout active, left) and validation loss (right) for the six learning
rates. Lines are means over three seeds and bands span the min–max across seeds.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for h, color in zip(SWEEPS["learning_rate"], palette(6)):
    band(axes[0], cfg_id(h), "train_loss", color, f"lr = {h.lr:g}")
    band(axes[1], cfg_id(h), "val_loss", color, f"lr = {h.lr:g}")
for ax in axes:
    loss_axes(ax)
# A run whose loss became NaN has no line to draw, so say so on the plot.
diverged = [h.lr for h in SWEEPS["learning_rate"] if runs[runs.cfg == cfg_id(h)].train_loss.isna().any()]
if diverged:
    for ax in axes:
        ax.text(0.03, 0.97, "diverged (loss became NaN): lr = " + ", ".join(f"{v:g}" for v in diverged),
                transform=ax.transAxes, va="top", fontsize=8, color="tab:red")
axes[0].set(title="Training loss (dropout on)", xlabel="epoch", ylabel="cross-entropy loss (log scale)")
axes[1].set(title="Validation loss", xlabel="epoch")
axes[1].legend(loc="center right", title="SGD + momentum 0.9\nbatch 64, dropout 0.25",
               fontsize=8, title_fontsize=8)
fig.suptitle("Figure 1. Learning-rate sweep", x=0.06, ha="left", fontweight="bold")
plt.tight_layout()
save(fig, "fig1_learning_rate.png")

### Figure 2: batch size, with and without scaling the learning rate

Top row: learning rate held at 0.01 while batch size changes. Bottom row: learning rate scaled
in proportion to batch size ($\text{lr} = 0.01 \times B / 64$).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5), sharex=True, sharey=True)
for row, sweep in enumerate(["batch_size", "batch_size_scaled_lr"]):
    for h, color in zip(SWEEPS[sweep], palette(4)):
        label = f"B = {h.batch_size}, lr = {h.lr:g}"
        band(axes[row, 0], cfg_id(h), "train_loss", color, label)
        band(axes[row, 1], cfg_id(h), "val_loss", color, label)
    axes[row, 1].legend(fontsize=8)
for ax in axes.flat:
    loss_axes(ax, top=1.2)
axes[0, 0].set(title="Training loss (dropout on)", ylabel="fixed lr = 0.01\ncross-entropy loss (log)")
axes[0, 1].set(title="Validation loss")
axes[1, 0].set(xlabel="epoch", ylabel="lr = 0.01 × B / 64\ncross-entropy loss (log)")
axes[1, 1].set(xlabel="epoch")
fig.suptitle("Figure 2. Batch-size sweep: fixed vs. linearly scaled learning rate", x=0.06, ha="left",
             fontweight="bold")
plt.tight_layout()
save(fig, "fig2_batch_size.png")

### Figure 3: dropout, and what the training curve hides

Left: the usual picture, training accuracy with dropout active vs. validation accuracy. Right:
the same models with the training subset re-scored in eval mode (the probe). The vertical
distance between a dashed line and the solid line of the same colour is the generalisation gap.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
colors = palette(4)
for h, color in zip(SWEEPS["dropout"], colors):
    cid = cfg_id(h)
    band(axes[0], cid, "train_acc", color, ls="--")
    band(axes[0], cid, "val_acc", color)
    band(axes[1], cid, "probe_acc", color, ls="--")
    band(axes[1], cid, "val_acc", color)
axes[0].set(title="As usually logged", xlabel="epoch", ylabel="accuracy")
axes[1].set(title="Training subset re-scored in eval mode", xlabel="epoch")
axes[0].set_ylim(0.80, 1.0)
for ax, train_label in zip(axes, ["train (dropout on, during epoch)", "train probe (eval mode)"]):
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    handles = [Line2D([], [], color=col, lw=2, label=f"p = {h.dropout:g}")
               for h, col in zip(SWEEPS["dropout"], colors)]
    handles += [Line2D([], [], color="k", ls="--", label=train_label),
                Line2D([], [], color="k", ls="-", label="validation")]
    ax.legend(handles=handles, fontsize=8, loc="lower right", ncol=2)
fig.suptitle("Figure 3. Dropout sweep (lr 0.01, batch 64)", x=0.06, ha="left", fontweight="bold")
plt.tight_layout()
save(fig, "fig3_dropout.png")

## 8. Choose on validation, then test once

The configuration with the highest mean final-epoch **validation** accuracy is selected. Only
then is the test set used: once for the selected configuration and once for the baseline as a
reference. The test numbers play no part in the choice.

In [ ]:
best_cfg = agg.val_acc.idxmax()
print(f"selected on validation: {best_cfg}  (val acc {agg.loc[best_cfg, 'val_acc']:.4f})")


@torch.no_grad()
def test_accuracy(cid, seed):
    model = TrackedClassifier(UNIQUE[cid])
    model.load_state_dict(torch.load(weight_file(cid, seed), map_location="cpu"))
    model.eval()
    x, y = dm.test_ds.tensors
    return (model(x).argmax(1) == y).float().mean().item()


rows = []
for cid in dict.fromkeys([best_cfg, cfg_id(BASE)]):
    accs = [test_accuracy(cid, s) for s in SEEDS]
    rows.append(dict(config=cid, role="selected" if cid == best_cfg else "baseline",
                     val_acc=agg.loc[cid, "val_acc"], test_acc=np.mean(accs),
                     test_acc_sd=np.std(accs, ddof=1)))
test_table = pd.DataFrame(rows)
test_table.to_csv(RESULTS / "test.csv", index=False)
print(test_table.to_string(index=False))

## 9. Download the outputs

Zips the result tables, the run log and the three figures (the weights stay behind; they are
only needed for section 8). In Colab the download starts automatically. Also save this notebook
with its outputs: *File → Download → Download .ipynb*.

In [ ]:
import zipfile

bundle = OUT / "hyperparameters_outputs.zip"
members = [*sorted(RESULTS.glob("*.csv")), PROGRESS, *sorted(FIGURES.glob("*.png"))]
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for f in members:
        z.write(f, f.relative_to(OUT))
print(f"{bundle} ({bundle.stat().st_size / 1e6:.2f} MB):", ", ".join(str(f.relative_to(OUT)) for f in members))

if IN_COLAB:
    from google.colab import files
    files.download(str(bundle))